# PulmoScan AI — Train on Google Colab

Hybrid **3D CNN + Swin Transformer** for lung-nodule malignancy risk prediction.

**Before running:** Runtime → Change runtime type → **GPU** (T4 / L4 / A100).

Your processed LIDC layout (from `PREPROCESSING_README.md`):
```
DATA_ROOT/
  volumes/*.nii.gz
  metadata/physical_nodules/*.json
  manifests/dataset_manifest.json
  manifests/patient_splits.json
```

## 1) Install the package

Option A: clone from git (set `REPO_URL`).  
Option B: upload a zip of this repo to Colab and unzip to `/content/pulmoscan`.

In [ ]:
from pathlib import Path
import os

REPO_URL = ""  # e.g. https://github.com/<you>/<repo>.git
WORK = Path("/content/pulmoscan")

if REPO_URL.strip():
    !git clone {REPO_URL} /content/pulmoscan
elif not (WORK / "pyproject.toml").exists():
    from google.colab import files
    import zipfile
    print("Upload project zip…")
    uploaded = files.upload()
    zname = next(iter(uploaded))
    WORK.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zname, "r") as zf:
        zf.extractall("/content")
    # If zip extracted a nested folder, try to locate pyproject
    if not (WORK / "pyproject.toml").exists():
        hits = list(Path("/content").rglob("pyproject.toml"))
        assert hits, "Could not find pyproject.toml after upload"
        WORK = hits[0].parent

os.chdir(WORK)
print("Working in", Path.cwd())
!pip -q install -e "."

## 2) Mount Drive and set `DATA_ROOT`

Upload your ~3 GB `data/processed` folder to Google Drive, then set the path below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_ROOT = "/content/drive/MyDrive/pulmoscan/data/processed"  # <-- edit me
manifest = Path(DATA_ROOT) / "manifests" / "dataset_manifest.json"
assert manifest.exists(), f"Missing {manifest}"
print("Data OK:", DATA_ROOT)

## 3) Optional synthetic smoke train (sanity check)

In [ ]:
USE_SYNTHETIC_SMOKE = False
if USE_SYNTHETIC_SMOKE:
    !python -m pulmoscan.data.synthetic --out /content/synth --n-train 8 --n-val 2 --n-test 2
    !python -m pulmoscan.classification.train --data-root /content/synth --epochs 2 --batch-size 2 --device auto --output-dir /content/ckpt_smoke

## 4) Train Hybrid CNN + Swin on LIDC

Label rule (defaults): physical nodules, mean malignancy ≥4 → malignant, ≤2 → benign, exclude 3s.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

EPOCHS = 30
BATCH_SIZE = 4
OUTPUT_DIR = "/content/drive/MyDrive/pulmoscan/artifacts/checkpoints"

!mkdir -p "{OUTPUT_DIR}"
!python -m pulmoscan.classification.train \
  --data-root "{DATA_ROOT}" \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --device cuda \
  --output-dir "{OUTPUT_DIR}"

## 5) Evaluate on test split

In [ ]:
!python -m pulmoscan.evaluation.evaluate \
  --data-root "{DATA_ROOT}" \
  --checkpoint "{OUTPUT_DIR}/best.pt" \
  --split test \
  --device cuda \
  --out "{OUTPUT_DIR}/test_metrics.json"

import json
from IPython.display import display
display(json.load(open(f"{OUTPUT_DIR}/test_metrics.json")))

## 6) Download checkpoint for local Gradio demo

In [ ]:
from google.colab import files
files.download(f"{OUTPUT_DIR}/best.pt")
print("Done. Locally: PULMOSCAN_CHECKPOINT=./best.pt python -m pulmoscan.inference.demo_app")